In [ ]:
import os
import re
import uuid
from functools import lru_cache
import yt_dlp #to download videos from youtube
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv # to load environment variables from a .env file
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# Load environment variables once, when the file is imported (works for the script AND for an API)
load_dotenv("key.env")
gemini_key = os.getenv("GEMINI_API_KEY")
if gemini_key:
    os.environ["GOOGLE_API_KEY"] = gemini_key


# ==========================================
#  SHARED RESOURCES 
#  built ONCE and reused for every question
# ==========================================

embedding_model = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')

prompt = PromptTemplate(
    template="""
    You are an expert AI assistant. Your task is to provide a clear, accurate, and concise answer to the user's question.

    Instructions:
    - Primary Answer: Base your main answer on the provided Context. Always share the content source (URL or title) provided in the context.
    - Handling Incomplete Data: If the context does not contain enough information to fully answer the question, you may provide additional relevant details from your general knowledge.
    - Outsourcing Disclaimer: If you include information that is NOT found in the context, you MUST explicitly state: "Note: This part is outsourcing from external knowledge/websites to give you the full content knowledge."
    - Format: Use bullet points for readability and keep the answer concise.

    Context:
    {context}

    Question: {query}
    """,
    input_variables=['context', 'query']
)

@lru_cache(maxsize=1)
def get_llm():
    # created on first use (so a missing key gives a clear error), then reused
    if not os.getenv("GOOGLE_API_KEY"):
        raise RuntimeError("GEMINI_API_KEY is not set. Add it to key.env or to your environment/secrets.")
    return ChatGoogleGenerativeAI(
        model="gemini-2.5-flash", 
        temperature=0.3, 
        max_tokens=1024
    )

vector_store_cache = {}   # persist_directory -> loaded FAISS index
rag_chain_cache = {}      # persist_directory -> ready-to-use chain


# ==========================================
# one folder per collection (instead of one shared ./faiss_db)
# ==========================================

def get_persist_directory(collection_id : str) -> str:
    # strict check so nobody can pass something like "../../secret" as an id
    if not re.fullmatch(r'[0-9a-f]{32}', collection_id):
        raise ValueError("Invalid collection_id.")
    return f"./faiss_db/{collection_id}"


# ==========================================
# DATA INGESTION 
# ==========================================

def load_youtube_videos(playlist_url : str) -> list: 
    video_urls= []
    ydl_opts = {
        'extract_flat' : True,  #extract video urls without downloading the videos
        'quiet' : True,  #suppress output
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            playlist_info = ydl.extract_info(playlist_url, download=False)
            if 'entries' in playlist_info:
                for entry in playlist_info['entries']:
                    if entry:
                        vid_id = entry.get('id')  ## yt-dlp 'extract_flat' usually stores the video ID under 'id'
                        if vid_id:
                            video_url = f"https://www.youtube.com/watch?v={vid_id}"
                            video_urls.append(video_url)
                        
        except Exception as e:
            print(f"Error loading playlist: {playlist_url}. Error: {e}")

    return video_urls


# ================================
# Data Processing
# ================================

import requests
import http.cookiejar

def load_videos(video_urls_list):
    loaded_docs = []
    # Check if we actually found videos
    if not video_urls_list:
        print("No video URLs found to process.")
        return loaded_docs
    
    
    session = requests.Session()
    try:
        cookie_jar = http.cookiejar.MozillaCookieJar('cookies.txt')
        cookie_jar.load(ignore_discard=True, ignore_expires=True)
        session.cookies.update(cookie_jar)
    except Exception as e:
        print(f"Warning: Could not load cookies: {e}")

    api = YouTubeTranscriptApi(http_client=session)
   
    for video_url in video_urls_list:
        try:      # Extract video ID from URL
            if "v=" in video_url:
                video_id = video_url.split("v=")[1].split("&")[0] # Extract video ID from URL
            elif "youtu.be/" in video_url:                         # short links like https://youtu.be/<id>
                video_id = video_url.split("youtu.be/")[1].split("?")[0]
            else:
                print(f"Invalid video URL format: {video_url}")
                continue
              
            fetched_transcript = api.fetch(
                video_id,
                languages=["en", "hi", "en-IN"]
            )
            transcript = [{"text": snippet.text} for snippet in fetched_transcript]
                        
            full_transcript_text = " ".join([item['text'] for item in transcript])
            doc =  Document(page_content = full_transcript_text,
                            metadata = {'source': video_url,
                                        'pages': 1,
                                        'title': f'Transcript for {video_id}',
                                        'author': 'YouTube'
                                        })
            loaded_docs.append(doc)
            print(f"Successfully loaded text for: {video_url}")  
            
        except Exception as e:
            print(f"Skipping video {video_url} due to error (e.g. no English/hindi transcripts): {e}")
    
    print('='*50)
    return loaded_docs


#playlist_url = 'https://www.youtube.com/playlist?list=PLZoTAELRMXVM8Pf4U67L4UuDRgV4TNX9D'
def process_youtube_videos(my_url : str):
    # checking if url is a playlist or a single video
    if 'list=' in my_url:
        video_urls = load_youtube_videos(my_url)
    else:
        print('Single video detected!')
        video_urls = [my_url]         

    all_loaded_docs = load_videos(video_urls)
    print(f"\nTotal Documents Loaded: {len(all_loaded_docs)}")

    return all_loaded_docs

#all_loaded_docs_withtrans = process_youtube_videos(playlist_url)


# ==========================================
# CHUNKING & EMBEDDING
# ==========================================

def generate_embedding(all_loaded_docs, persist_directiory = './faiss_db'):
    if not all_loaded_docs:
            print("No documents to embed!")
            return None
    #chunking
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    text_chunks = text_splitter.split_documents(all_loaded_docs)
    print(f"Split {len(all_loaded_docs)} original documents into {len(text_chunks)} smaller chunks.")
    print(text_chunks[0])
    
    print('*'*50)

    # uses the shared embedding_model (loaded once at the top) instead of loading it again here
    vectorstore = FAISS.from_documents(
            documents= text_chunks,
            embedding= embedding_model
            )
    # Save the FAISS database locally
    vectorstore.save_local(persist_directiory)
    vector_store_cache[persist_directiory] = vectorstore   # FIX 4: keep it in memory, no need to reload it from disk
    print(f"Success! Your text is now embedded and stored locally in a Vector Database named {persist_directiory}.")
    # vectorstore.get_by_ids([''])
    return vectorstore


#my_vector_database = generate_embedding(all_loaded_docs_withtrans,"./faiss_db")

#my_vector_database


def get_vector_store(persist_directory : str):
    # load the saved index from disk only the FIRST time, then reuse it from memory
    if persist_directory not in vector_store_cache:
        if not os.path.exists(persist_directory):
            raise FileNotFoundError(f"No vector database found at {persist_directory}")
        vector_store_cache[persist_directory] = FAISS.load_local(
            folder_path= persist_directory,
            embeddings=embedding_model,
            allow_dangerous_deserialization= True   # Required by LangChain to load local pickle files
        )
    return vector_store_cache[persist_directory]


def retriever_func(query: str, persist_directory : str = './faiss_db') ->list:
    vector_store = get_vector_store(persist_directory)
    #finding the top 4 relevent vectors from database related to query
    retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})
    docs = retriever.invoke(query)
    return docs


# query = 'what is LLM?'
# reterived_docs = retriever_func(query)   


# ==========================================
# RAG GENERATION WITH LCEL (RUNNABLES)
# ==========================================

def format_doc(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown Website')}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    )


def get_rag_chain(persist_directory : str):
    # the chain is built once per database and reused for every question
    if persist_directory not in rag_chain_cache:
        # Parallel retrieval chain
        setup_and_retrieval = RunnableParallel(
                context = RunnableLambda(lambda query: retriever_func(query, persist_directory)) | RunnableLambda(format_doc),
                query = RunnablePassthrough()
                )
        #full execution
        rag_chain_cache[persist_directory] = setup_and_retrieval | prompt | get_llm() | StrOutputParser()
    return rag_chain_cache[persist_directory]


def generate_answers(query, persist_directory = './faiss_db'):  
    rag_chain = get_rag_chain(persist_directory)

    # Invoke the LLM
    print(f"Asking the LLM: {query}\n")
    response = rag_chain.invoke(query)

    # Print and return the clean output
    print("$$$ FINAL ANSWER $$$")
    
    
    return response


# ==========================================
# FINAL EXECUTION
# ==========================================
if __name__ == "__main__":
    if not gemini_key:
        print("WARNING: Please ensure GEMINI_API_KEY is set in your key.env file.")
        raise SystemExit   # stop now instead of failing at the first question, after all the ingestion work

    # Ask the user for the YouTube URL
    user_url = input("Enter the YouTube Video or Playlist URL: ").strip()
    print("\nLoading transcripts... This might take a moment.")

    # Process and download transcripts
    all_loaded_docs_withtrans = process_youtube_videos(user_url)

    if not all_loaded_docs_withtrans:
        print("No transcripts could be loaded for this URL. Exiting.")
        raise SystemExit   # before: the script went on to the Q&A loop and crashed on a missing database

    # Generate Vector DB (every run gets its own folder)
    print("\nCreating the Vector Database...")
    collection_id = uuid.uuid4().hex
    persist_directory = get_persist_directory(collection_id)
    my_vector_database = generate_embedding(all_loaded_docs_withtrans, persist_directory)

 
    print("\n" + "="*50)
    print("Database Ready! Ask questions about the video(s).")
    print("Type 'exit' or 'quit' to stop.")
    print("="*50 + "\n")

    # Start the Interactive Q&A While Loop
    while True:
            # Ask for the user's question
        query = input("Your Question: ").strip()
        
        #pass the query and the chunks directly into generator function
        if query.lower() in ['exit', 'quit', 'q']:
            print("Ending the session. Goodbye!")
            break

        if not query:
            continue

        final_answer = generate_answers(query, persist_directory)   
        print(final_answer)                                          
        print('*'*50)

C:\Users\sharddha\AppData\Local\Temp\ipykernel_19928\871376109.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
d:\rag_project\basic_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1218.88it/s]



Loading transcripts... This might take a moment.
Successfully loaded text for: https://www.youtube.com/watch?v=fZM3oX4xEyg
Successfully loaded text for: https://www.youtube.com/watch?v=MykcjWPJ6T4
Successfully loaded text for: https://www.youtube.com/watch?v=adPi3a8fq4c
Successfully loaded text for: https://www.youtube.com/watch?v=MMS04bku3FE
Successfully loaded text for: https://www.youtube.com/watch?v=hSO4zjeQO48
Successfully loaded text for: https://www.youtube.com/watch?v=Chl-cRcwVpA
Successfully loaded text for: https://www.youtube.com/watch?v=FepDo-0DrSo
Successfully loaded text for: https://www.youtube.com/watch?v=NebOSOTp-zA

Total Documents Loaded: 8

Creating the Vector Database...
Split 8 original documents into 300 smaller chunks.
page_content='Hello all, my name is Krishna and welcome to my YouTube channel. So guys, I am super excited to start this new series on one of the most important technique which is right now being used in genative AI and agentic AI field that is n

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Asking the LLM: what is cv

$$$ FINAL ANSWER $$$
The provided context does not contain information about "cv".

Note: This part is outsourcing from external knowledge/websites to give you the full content knowledge.

"CV" typically stands for:
*   **Curriculum Vitae**: A detailed document outlining a person's academic and professional history, often used for job applications, especially in academia or research.
*   **Computer Vision**: A field of artificial intelligence that enables computers to "see" and interpret visual information from the world, such as images and videos.
**************************************************
Ending the session. Goodbye!
